# RoboBarista v2 — LeRobot × RoboCasa auf Google Colab

**Ein frischer Start, ohne die Fehlersuche von gestern wiederholen zu müssen.** Dieses Notebook installiert LeRobot 0.6.1, RoboCasa365 aus deinem `barista-dev`-Branch und **den robosuite-Commit aus LeRobots offizieller RoboCasa-Testumgebung**. Danach richtet es **alle vier erforderlichen Assetpakete** ein, lädt die bestehende SmolVLA-Policy und testet die RoboCasa-Simulation.

**Google Colab Free:** Wähle `Laufzeit → Laufzeittyp ändern → T4-GPU`. Dann die Zellen von oben nach unten ausführen. **Bei einer vollständig neuen Laufzeit muss Python neu installiert werden**. Mit Google-Drive-Cache können ZIP-Assets und Hugging-Face-Modellgewichte zwischen Sitzungen erhalten bleiben; die entpackten lokalen Simulationsdateien und die Python-Umgebung werden erneut angelegt.

**Wichtig:** Ein erfolgreiches `CloseFridge` beweist nur, dass die vorhandene Policy in der Simulation läuft. **Variable Startpositionen**, **gezielte Störungen während der Bewegung** und **mm-genaue Platzierung mehrerer Objekte** sind anschließende Experimente, keine zugesicherte Eigenschaft des Modells. Wir entwickeln keine Greif-Trajektorien von Hand.

Quellen: [Offizielle LeRobot-RoboCasa-Anleitung](https://huggingface.co/docs/lerobot/v0.6.1/robocasa), [LeRobot RoboCasa CI-Konfiguration](https://github.com/huggingface/lerobot/blob/v0.6.1/docker/Dockerfile.benchmark.robocasa), [vortrainierte Policy](https://huggingface.co/lerobot/smolvla_robocasa), [offizielle Texturpakete](https://huggingface.co/datasets/robocasa/robocasa-assets), [Lightwheel-Fixtures-Spiegel](https://huggingface.co/datasets/IIFAN/robocasa365-lightwheel-fixtures), [Lightwheel-Objekte-Spiegel](https://huggingface.co/datasets/IIFAN/robocasa365-lightwheel-objects).

Die vier ZIP-Archive benötigen **rund 3,05 GB** im dauerhaften Cache. Zusätzlich benötigen die Modelle einige GB. Prüfe vorher deinen verfügbaren Google-Drive-Speicher. Die beiden Lightwheel-Repositories sind dokumentierte Community-Spiegel; alle Archive werden auf konkrete Dateigröße und SHA-256 geprüft.


## 1. GPU und Speicherort festlegen

Mit `USE_GOOGLE_DRIVE = True` musst du **einmal den Google-Drive-Zugriff bestätigen**. Im Drive-Ordner `MyDrive/RoboBarista` bleiben die ZIP-Archive, Hugging-Face-Modellgewichte und Simulationsergebnisse erhalten. Beim nächsten Colab-Start wird nur die Software neu installiert und die Assets werden lokal entpackt.

Wenn du keinen freien Drive-Speicher hast, setze `USE_GOOGLE_DRIVE = False`. Dann funktioniert das Notebook ebenfalls, aber die großen Downloads müssen nach jeder vollständig neuen Laufzeit wiederholt werden.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

USE_GOOGLE_DRIVE = True  # Bei Platzmangel auf False setzen

assert shutil.which("nvidia-smi"), "Bitte Colab auf T4-GPU umstellen und neu verbinden."
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
print("Freier lokaler Speicher:", round(shutil.disk_usage("/content").free / 1024**3, 1), "GiB")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = Path("/content/drive/MyDrive/RoboBarista")
else:
    PERSIST = Path("/content/RoboBarista_cache")

ARCHIVES = PERSIST / "asset_archives"
HF_CACHE = PERSIST / "huggingface_cache"
RESULTS = PERSIST / "results"
for folder in (ARCHIVES, HF_CACHE, RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

print("Dauerhafter Cache:" if USE_GOOGLE_DRIVE else "Temporärer Cache:", PERSIST)
print("Python des Notebooks:", subprocess.check_output(["python", "--version"], text=True).strip())


## 2. Projektversionen herunterladen

**Wichtig:** Nicht `robosuite v1.5.2` verwenden! Dieser Stand führte bei uns zum Fehler `load_model_on_init`. Der folgende Commit ist exakt derjenige aus der offiziellen LeRobot-RoboCasa-Dockerdatei. Die RoboCasa-Quelle stammt aus deinem eigenen GitHub-Branch.

In [ ]:
from pathlib import Path
import subprocess

repos = [
    ("https://github.com/Coffee-Tobi/robocasa.git", "/content/robocasa", "barista-dev"),
    ("https://github.com/huggingface/lerobot.git", "/content/lerobot", "v0.6.1"),
    ("https://github.com/ARISE-Initiative/robosuite.git", "/content/robosuite", "master"),
]
for url, folder, ref in repos:
    if not (Path(folder) / ".git").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", ref, url, folder], check=True)
    else:
        print("Bereits vorhanden:", folder)

ROBOSUITE_SHA = "aaa8b9b214ce8e77e82926d677b4d61d55e577ab"
rs = "/content/robosuite"
changes = subprocess.check_output(["git", "-C", rs, "status", "--porcelain"], text=True).strip()
assert not changes, "Lokale Änderungen im robosuite-Verzeichnis; bitte vor Checkout prüfen: " + changes
actual = subprocess.check_output(["git", "-C", rs, "rev-parse", "HEAD"], text=True).strip()
if actual != ROBOSUITE_SHA:
    subprocess.run(["git", "-C", rs, "fetch", "--depth", "1", "origin", ROBOSUITE_SHA], check=True)
    subprocess.run(["git", "-C", rs, "checkout", "--detach", "FETCH_HEAD"], check=True)
assert subprocess.check_output(["git", "-C", rs, "rev-parse", "HEAD"], text=True).strip() == ROBOSUITE_SHA

for _, folder, _ in repos:
    commit = subprocess.check_output(["git", "-C", folder, "rev-parse", "--short", "HEAD"], text=True).strip()
    print(folder, "→", commit)


## 3. Python 3.12 und alle nötigen Pakete installieren

Colab verwendet im Notebook selbst Python 3.13; wir installieren **LeRobot und RoboCasa in einer separaten Python-3.12-Umgebung** unter `/content/robobarista-py312`. Die erste Installation benötigt mehrere GB und kann länger dauern.

Das Setup enthält von Anfang an `lerobot[smolvla,dataset,evaluation]` – damit tritt unser früherer `ImportError: datasets` nicht mehr auf. RoboCasa wird mit `--no-deps` eingebunden, damit sein alter LeRobot-0.3.3-Pin die neue LeRobot-Version nicht überschreibt.

In [ ]:
import os, subprocess
from pathlib import Path

MICRO = "/content/bin/bin/micromamba"
if not Path(MICRO).exists():
    subprocess.run(
        "mkdir -p /content/bin && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/bin bin/micromamba",
        shell=True, check=True
    )
ENV_DIR = "/content/robobarista-py312"
if not Path(ENV_DIR, "bin/python").is_file():
    subprocess.run([MICRO, "create", "-y", "-p", ENV_DIR, "-c", "conda-forge", "python=3.12", "pip"], check=True)
PY = ENV_DIR + "/bin/python"
print(subprocess.check_output([PY, "--version"], text=True).strip())

install_env = os.environ.copy()
install_env["PIP_NO_CACHE_DIR"] = "1"
install_env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"

def pip_install(*packages):
    print("\nInstalliere:", *packages, flush=True)
    subprocess.run([PY, "-m", "pip", "install", "--no-cache-dir", *packages], env=install_env, check=True)

# Offizielle Policy + ihre Datensatz- und Evaluierungs-Abhängigkeiten.
pip_install("-e", "/content/lerobot[smolvla,dataset,evaluation]")
pip_install("-e", "/content/robosuite")

# Aus LeRobots offizieller RoboCasa-Docker-Konfiguration (v0.6.1).
pip_install(
    "numpy==2.2.5", "numba==0.61.2", "scipy==1.15.3", "mujoco==3.3.1",
    "pygame==2.6.1", "Pillow==12.2.0", "opencv-python==4.13.0.92",
    "pyyaml==6.0.3", "pynput==1.8.1", "tqdm==4.67.3",
    "termcolor==3.3.0", "imageio==2.37.3", "h5py==3.16.0",
    "lxml==6.0.4", "hidapi==0.14.0.post4", "tianshou==0.4.10",
    "gymnasium==1.2.3",
)
pip_install("--no-deps", "-e", "/content/robocasa")
print("✅ Alle Python-Pakete installiert.")


## 4. Konfiguration und GPU-Kompatibilität prüfen

Dies ersetzt die interaktive `setup_macros.py`-Zelle, die früher in Colab mit einem Fehler abgebrochen ist. Bereits bestehende Einstellungen werden **nicht überschrieben**. Zudem kontrollieren wir den erforderlichen robosuite-Parameter `load_model_on_init`.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

for folder in ("/content/robocasa/robocasa", "/content/robosuite/robosuite"):
    source = Path(folder) / "macros.py"
    private = Path(folder) / "macros_private.py"
    if source.is_file() and not private.exists():
        shutil.copy2(source, private)
        print("Konfiguration erstellt:", private)
    elif private.exists():
        print("Konfiguration bereits vorhanden:", private)

TEST_ENV = os.environ.copy()
TEST_ENV.update({
    "MUJOCO_GL": "egl",
    "PYOPENGL_PLATFORM": "egl",
    "OPENBLAS_NUM_THREADS": "1",
    "HF_HOME": str(HF_CACHE),
    "HF_HUB_DOWNLOAD_TIMEOUT": "300",
})
check = """
import sys, importlib, inspect
import torch
from robosuite.environments.manipulation.manipulation_env import ManipulationEnv
for name in ('torch', 'mujoco', 'robosuite', 'robocasa', 'lerobot', 'datasets', 'huggingface_hub'):
    importlib.import_module(name)
    print(name, 'OK', flush=True)
assert 'load_model_on_init' in inspect.signature(ManipulationEnv.__init__).parameters
assert torch.cuda.is_available(), 'CUDA nicht verfügbar'
print('✅ Python', sys.version.split()[0], '| GPU:', torch.cuda.get_device_name(0), flush=True)
"""
subprocess.run([PY, "-u", "-c", check], env=TEST_ENV, check=True)


## 5. **Alle** Küchen-Assets vollständig einrichten

**Dieser Schritt ersetzt die bisherigen Abschnitte 6, 6b, 6c, 6d und 6e.** Er lädt vier fest definierte Archive (Basistexturen, generierte Texturen, Lightwheel-Fixtures und Lightwheel-Objekte) von Hugging Face. **Jedes ZIP wird vollständig per SHA-256 und Dateigröße geprüft.** Danach wird der genaue Archivpfad geprüft, in den richtigen RoboCasa-Ordner entpackt und **jede Archivdatei** auf Existenz und Dateigröße geprüft.

Die ZIPs werden bei aktivem Google Drive in `MyDrive/RoboBarista/asset_archives` gespeichert. Nach einem Reset wird nicht erneut heruntergeladen, sondern aus diesen Archiven in die frische Colab-Umgebung entpackt. Keine 30-GB-Objaverse-Sammlung; die Policy verwendet die Lightwheel-Registry.

**Dieser Vorgang kann beim ersten Mal länger dauern** (mehrere GB Download und viele Tausend Dateien zum Entpacken).

In [ ]:
import os, subprocess
from pathlib import Path

ASSET_SETUP = r'''
from pathlib import Path
from hashlib import sha256
from shutil import disk_usage
from zipfile import ZipFile
from huggingface_hub import hf_hub_download
import sys

archive_cache = Path(sys.argv[1])
assets = Path("/content/robocasa/robocasa/models/assets")
archive_cache.mkdir(parents=True, exist_ok=True)

# Größen und SHA-256 aus den HF-Commit-Metadaten bzw. den dokumentierten
# byte-identischen Lightwheel-Spiegeln. KEINE großen Objaverse-Archive.
specs = [
    {
        "title": "Basistexturen",
        "repo": "robocasa/robocasa-assets",
        "revision": "1b92c3d02ca4354984fec961357db0bff7b32166",
        "filename": "textures.zip",
        "bytes": 543180421,
        "sha256": "ba3a4dcef96e199cbd601c7cdb7eb46077fb6672611b7ac74c25a191bf0be392",
        "entry_full": "textures/tiles/orange_square_tiles.png",
        "entry_short": "tiles/orange_square_tiles.png",
        "dest_full": assets,
        "dest_short": assets / "textures",
    },
    {
        "title": "Generierte Texturen",
        "repo": "robocasa/robocasa-assets",
        "revision": "1b92c3d02ca4354984fec961357db0bff7b32166",
        "filename": "generative_textures.zip",
        "bytes": 1184675536,
        "sha256": "bc05095af306222d9a799a9b0b5b16662aa546998622ee8fd3072566ea4c31ec",
        "entry_full": "generative_textures/wall/tex027.png",
        "entry_short": "wall/tex027.png",
        "dest_full": assets,
        "dest_short": assets / "generative_textures",
    },
    {
        "title": "Lightwheel-Fixtures",
        "repo": "IIFAN/robocasa365-lightwheel-fixtures",
        "revision": None,
        "filename": "fixtures_lightwheel.zip",
        "bytes": 529745465,
        "sha256": "62c1c87776edda9b7c42dda7b00781a1e5a7f90acc3e49d06c39c9545cbd6c3e",
        "entry_full": "fixtures/windows/Window052/model.xml",
        "entry_short": "windows/Window052/model.xml",
        "dest_full": assets,
        "dest_short": assets / "fixtures",
    },
    {
        "title": "Lightwheel-Objekte",
        "repo": "IIFAN/robocasa365-lightwheel-objects",
        "revision": None,
        "filename": "objects_lightwheel.zip",
        "bytes": 791939656,
        "sha256": "a1db739b0e80fc4d56d4be27d89dba3aa60000e9648c5607874700e0f14b7955",
        "entry_full": "lightwheel/stool/Stool020/model.xml",
        "entry_short": "stool/Stool020/model.xml",
        "dest_full": assets / "objects",
        "dest_short": assets / "objects/lightwheel",
    },
]

markers = assets / ".robobarista_complete"
markers.mkdir(parents=True, exist_ok=True)

def check_sha(file):
    digest = sha256()
    with file.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for spec in specs:
    archive_path = archive_cache / spec["filename"]
    stamp = markers / (spec["filename"] + ".sha256")
    sentinel = spec["dest_full"] / spec["entry_full"]
    if stamp.is_file() and stamp.read_text().strip() == spec["sha256"] and sentinel.is_file():
        print("✅ Bereits vollständig entpackt:", spec["title"], flush=True)
        continue

    print("\nPaket:", spec["title"], flush=True)
    if not archive_path.is_file():
        print("  Lade einmalig nach:", archive_path, flush=True)
        kwargs = dict(
            repo_id=spec["repo"],
            repo_type="dataset",
            filename=spec["filename"],
            local_dir=str(archive_cache),
        )
        if spec["revision"] is not None:
            kwargs["revision"] = spec["revision"]
        downloaded = Path(hf_hub_download(**kwargs))
        assert downloaded.resolve() == archive_path.resolve(), (downloaded, archive_path)

    assert archive_path.stat().st_size == spec["bytes"], (
        f"Falsche Dateigröße bei {archive_path}. "
        "Download möglicherweise unvollständig; bitte die Ausgabe teilen."
    )
    actual_sha = check_sha(archive_path)
    assert actual_sha == spec["sha256"], (
        f"Falscher SHA-256 bei {archive_path}: {actual_sha}; nichts wird entpackt."
    )
    print("  ✅ Größe und SHA-256 geprüft.", flush=True)

    with ZipFile(archive_path) as z:
        names = set(z.namelist())
        if spec["entry_full"] in names:
            dest = spec["dest_full"]
        elif spec["entry_short"] in names:
            dest = spec["dest_short"]
        else:
            hits = [n for n in names if n.endswith(spec["entry_short"])][:10]
            raise RuntimeError(
                f"Erforderlicher Pfad fehlt in {spec['filename']}. Ähnliche: {hits}"
            )
        root = dest.resolve()
        for item in z.infolist():
            assert (root / item.filename).resolve().is_relative_to(root), (
                f"Unzulässiger ZIP-Pfad: {item.filename}"
            )
        needed = sum(item.file_size for item in z.infolist() if not item.is_dir())
        free = disk_usage("/content").free
        assert free > needed + 1024**3, (
            f"Nicht genug lokaler Speicher zum Entpacken von {spec['title']}."
        )
        print(f"  Entpacke {len(names)} Archiv-Einträge ...", flush=True)
        z.extractall(dest)
        missing = []
        for item in z.infolist():
            if not item.is_dir():
                f = dest / item.filename
                if not f.is_file() or f.stat().st_size != item.file_size:
                    missing.append(item.filename)
                    if len(missing) >= 5:
                        break
        assert not missing, (
            f"Entpackung unvollständig in {spec['title']}. Beispiele: {missing}"
        )
        stamp.write_text(spec["sha256"])
        print("  ✅ Vollständig eingerichtet:", spec["title"], flush=True)

for spec in specs:
    p = spec["dest_full"] / spec["entry_full"]
    assert p.is_file(), f"Erwartetes Asset fehlt: {p}"
    print("✅", spec["title"], "→", p, flush=True)
print("\n✅ Assetprüfung vollständig erfolgreich!", flush=True)
'''
script_path = Path("/content/robobarista_setup_assets.py")
script_path.write_text(ASSET_SETUP, encoding="utf-8")
subprocess.run(
    [PY, "-u", str(script_path), str(ARCHIVES)],
    env=TEST_ENV,
    check=True,
)


## 6. SmolVLA-RoboCasa-Modell **verifiziert** vorbereiten

**Die Korrektur für den Fehler `SafetensorError: header too large`:** Der bisherige Hugging-Face-Cache auf Google Drive enthielt offenbar keine lesbaren Modellgewichte. Wir prüfen deshalb die offizielle `model.safetensors`-Datei auf **exakte Größe (906.712.520 Bytes), SHA-256 und einen gültigen Safetensors-Header**, bevor die Simulation startet. Das verhindert, dass wir erneut minutenlang eine nicht lesbare Datei laden.

Die Datei wird als **echte Datei**, nicht als Hugging-Face-Cache-Symlink, im Drive-Ordner `MyDrive/RoboBarista/model_files/smolvla_robocasa` gesichert und pro Sitzung nach `/content/robobarista_policy/smolvla_robocasa` kopiert. Ein beschädigter Google-Drive-Hub-Cache wird nicht für den Download verwendet. Die Rohdateien bleiben beim nächsten Colab-Reset erhalten, sofern du Google Drive aktiviert hast.

Checkpoint: https://huggingface.co/lerobot/smolvla_robocasa/blob/main/model.safetensors · Gepinnte Repo-Revision: `38d38bf58e1217092c5a81418fcc8eef1a7c3a6f`. **Nichts wird trainiert.**

In [ ]:
import subprocess
from pathlib import Path

# Verifizierte Originaldatei (Hugging Face):
# https://huggingface.co/lerobot/smolvla_robocasa/blob/main/model.safetensors
# 906712520 Bytes, SHA256 96ae6628166a7dc1b41f8a2d92176c48e394988294b0b44755409acbb3f9bc2b
MODEL_LOCAL_DIR = Path("/content/robobarista_policy/smolvla_robocasa")
MODEL_DRIVE_DIR = PERSIST / "model_files" / "smolvla_robocasa"

policy_script = r'''
from pathlib import Path
from hashlib import sha256
from huggingface_hub import snapshot_download
from safetensors import safe_open
import shutil
import sys

saved = Path(sys.argv[1])
local = Path(sys.argv[2])
use_drive = sys.argv[3] == "1"
expected_bytes = 906712520
expected_hash = "96ae6628166a7dc1b41f8a2d92176c48e394988294b0b44755409acbb3f9bc2b"
required = ["config.json", "model.safetensors", "policy_preprocessor.json", "policy_postprocessor.json"]

def valid(path):
    if not all((path / item).is_file() for item in required):
        return False
    file = path / "model.safetensors"
    if file.stat().st_size != expected_bytes:
        print(f"Ungültige Modellgröße: {file.stat().st_size} statt {expected_bytes} Bytes", flush=True)
        return False
    h = sha256()
    with file.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    if h.hexdigest() != expected_hash:
        print("SHA-256 des Modells stimmt nicht.", flush=True)
        return False
    try:
        with safe_open(str(file), framework="pt", device="cpu") as sf:
            print(f"Safetensors-Header gültig ({len(sf.keys())} Tensoren).", flush=True)
    except Exception as e:
        print("Safetensors-Header ungültig:", repr(e), flush=True)
        return False
    return True

if valid(local):
    print("✅ Lokale Policy bereits geprüft.", flush=True)
elif use_drive and valid(saved):
    print("✅ Modell im Google Drive gültig. Kopiere es lokal ...", flush=True)
    shutil.copytree(saved, local, dirs_exist_ok=True)
    assert valid(local), "Kopie aus Google Drive beschädigt."
else:
    print("Gespeicherte Policy fehlt oder ist beschädigt. Lade das Original neu (ca. 907 MB) ...", flush=True)
    local.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id="lerobot/smolvla_robocasa",
        revision="f7af588d11e30b0069f200792fb285904b74c80b",
        local_dir=str(local),
        force_download=True,
        max_workers=2,
    )
    assert valid(local), "Auch der neue Download ist nicht gültig. Bitte hier stoppen und Ausgabe teilen."
    if use_drive:
        try:
            saved.mkdir(parents=True, exist_ok=True)
            for file in local.iterdir():
                if file.is_file():
                    shutil.copy2(file, saved / file.name)
            print("✅ Verifizierte Modelldateien im Drive gesichert:", saved, flush=True)
        except Exception as e:
            print("⚠️ Drive-Kopie nicht möglich; lokale Policy ist trotzdem lauffähig:", str(e), flush=True)

print("✅ Vollständig geprüfte Policy:", local, flush=True)
'''

# Wichtig: kein Download aus dem zuvor möglicherweise korrupten Google-Drive-HF-Cache.
fresh_env = TEST_ENV.copy()
fresh_env["HF_HOME"] = "/content/robobarista_policy_download_cache"
subprocess.run(
    [PY, "-u", "-c", policy_script,
     str(MODEL_DRIVE_DIR), str(MODEL_LOCAL_DIR),
     "1" if USE_GOOGLE_DRIVE else "0"],
    env=fresh_env,
    check=True,
)


## 6b. SmolVLM2-Basismodell separat prüfen und lokal bereitstellen

**Zweiter SHA-256-Fehler vom 20. Oktober:** Die SmolVLA-Policy kann geladen werden, aber ihr **externes** SmolVLM2-Vision-Language-Modell stürzt beim Lesen eines Safetensors-Headers ab. Die Ursache liegt in der beschädigten beziehungsweise falschen Gewichtsdatei des Basismodells im bisherigen Hugging-Face-Cache – nicht an den Küchen-Assets.

Wir verwenden deshalb eine **separate, lokale und überprüfte Kopie** von `HuggingFaceTB/SmolVLM2-500M-Video-Instruct`. Die originale `model.safetensors` ist etwa 2,03 GB groß; der SHA-256-Hash ist auf [Hugging Face](https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct/blob/eb8cd98a000e493c643b0cc72d15e43b0da834f2/model.safetensors) veröffentlicht. Wir überprüfen die Datei vor dem Evaluieren auf SHA-256 und einen gültigen Safetensors-Header. Die geprüften Dateien werden separat im Drive unter `RoboBarista/model_files/smolvlm2_base` gespeichert und in Colab lokal kopiert. **Nur die Basismodell-Dateien** werden bei Bedarf heruntergeladen. Weitere 2,03 GB Drive-Speicher sind nötig.

Der spätere Policy-Aufruf erhält `--policy.vlm_model_name=/content/robobarista_policy/smolvlm2_base`, sodass Transformers **nicht mehr den unzuverlässigen Drive-Hub-Cache benutzt**.

In [ ]:
import subprocess
from pathlib import Path

BASE_VLM_LOCAL_DIR = Path("/content/robobarista_policy/smolvlm2_base")
BASE_VLM_DRIVE_DIR = PERSIST / "model_files" / "smolvlm2_base"
base_script = "from pathlib import Path\nfrom hashlib import sha256\nfrom huggingface_hub import snapshot_download\nfrom safetensors import safe_open\nimport shutil\nimport sys\n\nsaved = Path(sys.argv[1])\nlocal = Path(sys.argv[2])\nuse_drive = sys.argv[3] == \"1\"\nrepo_id = \"HuggingFaceTB/SmolVLM2-500M-Video-Instruct\"\nrevision = \"eb8cd98a000e493c643b0cc72d15e43b0da834f2\"\nexpected_sha = \"b9bfd456c9472c0acd5719d6e514c4b859891af205ee1a736552fd3497b8b0c3\"\nrequired = [\n    \"model.safetensors\", \"config.json\", \"preprocessor_config.json\",\n    \"processor_config.json\", \"tokenizer.json\", \"tokenizer_config.json\",\n    \"special_tokens_map.json\", \"vocab.json\", \"merges.txt\",\n]\n\ndef valid(folder):\n    if not all((folder / name).is_file() for name in required):\n        return False\n    weight = folder / \"model.safetensors\"\n    if weight.stat().st_size < 1_000_000_000:\n        print(\"SmolVLM2-Datei ist zu klein:\", weight.stat().st_size, flush=True)\n        return False\n    digest = sha256()\n    with weight.open(\"rb\") as handle:\n        for data in iter(lambda: handle.read(8 * 1024 * 1024), b\"\"):\n            digest.update(data)\n    if digest.hexdigest() != expected_sha:\n        print(\"SmolVLM2-SHA256 falsch:\", digest.hexdigest(), flush=True)\n        return False\n    try:\n        with safe_open(str(weight), framework=\"pt\", device=\"cpu\") as f:\n            print(\"SmolVLM2 gültig:\", len(f.keys()), \"Tensoren.\", flush=True)\n    except Exception as err:\n        print(\"SmolVLM2-Safetensors-Header unlesbar:\", err, flush=True)\n        return False\n    return True\n\nif valid(local):\n    print(\"✅ SmolVLM2 bereits lokal überprüft.\", flush=True)\nelif use_drive and valid(saved):\n    print(\"✅ SmolVLM2 auf Drive verifiziert; kopiere lokal ...\", flush=True)\n    shutil.copytree(saved, local, dirs_exist_ok=True,\n                    ignore=shutil.ignore_patterns(\".cache\"))\n    assert valid(local), \"Kopie vom Drive nicht lesbar.\"\nelse:\n    print(\"SmolVLM2 fehlt oder ist beschädigt. Lade nur dieses Basismodell (~2,03 GB) ...\", flush=True)\n    local.mkdir(parents=True, exist_ok=True)\n    snapshot_download(\n        repo_id=repo_id,\n        revision=revision,\n        local_dir=str(local),\n        allow_patterns=[\n            \"*.safetensors\", \"*.json\", \"*.txt\",\n        ],\n        force_download=True,\n        max_workers=2,\n    )\n    assert valid(local), \"Download nicht intakt; bitte Ausgabe statt Evaluation schicken.\"\n    if use_drive:\n        try:\n            saved.mkdir(parents=True, exist_ok=True)\n            for name in required:\n                shutil.copy2(local / name, saved / name)\n            print(\"✅ Geprüftes SmolVLM2 in Google Drive gespeichert.\", flush=True)\n        except Exception as err:\n            print(\"⚠️ Drive-Sicherung fehlgeschlagen, aber lokale Dateien gültig:\", err, flush=True)\n\nprint(\"✅ SmolVLM2 bereit für die Simulation:\", local, flush=True)\n"

# Umgeht den bisher fehlerhaften HF_HOME-Cache direkt.
clean_env = TEST_ENV.copy()
clean_env["HF_HOME"] = "/content/robobarista_base_vlm_download_cache"

subprocess.run(
    [PY, "-u", "-c", base_script,
     str(BASE_VLM_DRIVE_DIR), str(BASE_VLM_LOCAL_DIR),
     "1" if USE_GOOGLE_DRIVE else "0"],
    env=clean_env,
    check=True,
)


## 6c. Offiziell korrigierte RoboCasa-Policy-Metadaten (16 Zustandswerte)

Beim lokalen Checkpoint waren bislang `config.json` und `policy_preprocessor.json` auf **6** Zustandswerte eingestellt, obwohl RoboCasa und der Trainingsdatensatz **16** liefern. Die Modellherausgeber haben dafür einen [offiziellen Fix](https://huggingface.co/lerobot/smolvla_robocasa/commit/f7af588d11e30b0069f200792fb285904b74c80b) veröffentlicht.

Wir ersetzen **nur diese zwei kleinen Metadatendateien** durch die offiziellen Fassungen aus dem Fix-Commit und überprüfen anschließend die Formen. Die unveränderten trainierten `model.safetensors`-Gewichte und das SmolVLM2-Basismodell werden hierbei **nicht erneut heruntergeladen**. Wenn Google Drive aktiviert ist, werden die korrigierten Metadaten auch dort gespeichert.

**Reihenfolge:** Zuerst Abschnitt 6, dann 6b und 6c, erst danach Abschnitt 7.

In [ ]:
import subprocess
metadata_script = "from pathlib import Path\nfrom huggingface_hub import hf_hub_download\nimport json\nimport shutil\nimport sys\n\nlocal = Path(sys.argv[1])\nsaved = Path(sys.argv[2])\nuse_drive = sys.argv[3] == \"1\"\nrepo = \"lerobot/smolvla_robocasa\"\nrev = \"f7af588d11e30b0069f200792fb285904b74c80b\"\nfiles = (\"config.json\", \"policy_preprocessor.json\")\n\ndef shape(path):\n    config = json.loads(path.read_text())\n    if path.name == \"config.json\":\n        return config[\"input_features\"][\"observation.state\"][\"shape\"]\n    for step in config[\"steps\"]:\n        if step.get(\"registry_name\") == \"normalizer_processor\":\n            return step[\"config\"][\"features\"][\"observation.state\"][\"shape\"]\n    raise RuntimeError(\"Normalizer-Prozessor fehlt in \" + str(path))\n\nfor name in files:\n    dest = local / name\n    if dest.is_file() and shape(dest) == [16]:\n        print(\"✅ Bereits korrigiert:\", name, flush=True)\n    else:\n        print(\"Lade offizielle 16D-Metadaten:\", name, flush=True)\n        downloaded = hf_hub_download(repo_id=repo, filename=name, revision=rev, force_download=True)\n        assert shape(Path(downloaded)) == [16], (\n            f\"Offizielles Fix-Commit liefert nicht 16D: {name}\"\n        )\n        shutil.copy2(downloaded, dest)\n        assert shape(dest) == [16], f\"Metadaten nach Kopieren falsch: {name}\"\n    if use_drive:\n        try:\n            saved.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(dest, saved / name)\n        except Exception as err:\n            print(\"⚠️ Konnte Metadaten nicht in Drive speichern:\", err, flush=True)\n\nfor name in files:\n    print(\"✅\", name, \"Zustandsdimension:\", shape(local / name), flush=True)\n"
metadata_env = TEST_ENV.copy()
metadata_env["HF_HOME"] = "/content/robobarista_metadata_download_cache"
subprocess.run(
    [PY, "-u", "-c", metadata_script,
     str(MODEL_LOCAL_DIR), str(MODEL_DRIVE_DIR),
     "1" if USE_GOOGLE_DRIVE else "0"],
    env=metadata_env,
    check=True,
)


## 7. Erste **echte Policy-Evaluation**: CloseFridge

Dies ist der Test, der gestern noch am unvollständigen Asset-Download scheiterte. Der Aufruf verwendet die Kamera-Schlüssel aus LeRobots offizieller RoboCasa-Anleitung und eine Episode auf der T4.

LeRobot lädt die in Abschnitt 6 SHA-256-geprüfte lokale Policy (nicht den möglicherweise beschädigten Drive-Hub-Cache). **Wichtig:** Bei einer bereits geöffneten Colab-Kopie müssen die neuen Abschnitte 6 und 7 übernommen werden; GitHub-Änderungen erscheinen dort nicht automatisch.

LeRobot legt bei normalen Evaluierungen standardmäßig Videos unter dem jeweiligen Ergebnisverzeichnis ab (sofern das Rendering funktioniert). Das vollständige Log wird gespeichert. Mit Google Drive werden nach dem Test auch Ergebnisse dorthin kopiert.

**Hinweis:** Dieser Test prüft erst die Integration. Für eine Statistik sind später mehr Episoden nötig.

**Vor dem Ausführen muss Abschnitt 6b erfolgreich beendet sein.** Erst dann existiert die lokale SmolVLM2-Kopie, die ausdrücklich als Modellquelle übergeben wird.


In [ ]:
import os, shutil, subprocess
from pathlib import Path
from datetime import datetime, timezone

RENAME_MAP = (
    '{"observation.images.robot0_agentview_left": "observation.images.camera1", '
    '"observation.images.robot0_eye_in_hand": "observation.images.camera2", '
    '"observation.images.robot0_agentview_right": "observation.images.camera3"}'
)

def evaluate(task, n_episodes=1, seed=1000):
    name = task + "_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    result_dir = Path("/content/robobarista_runs") / name
    result_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        ENV_DIR + "/bin/lerobot-eval",
        "--policy.path=" + str(MODEL_LOCAL_DIR),
        "--policy.vlm_model_name=" + str(BASE_VLM_LOCAL_DIR),
        "--env.type=robocasa", "--env.task=" + task,
        "--eval.batch_size=1", "--eval.n_episodes=" + str(n_episodes),
        "--eval.use_async_envs=false",
        "--policy.device=cuda",
        "--seed=" + str(seed),
        "--output_dir=" + str(result_dir),
        "--rename_map=" + RENAME_MAP,
    ]
    log_path = result_dir / "eval.log"
    # Nie den alten (evtl. beschädigten) HF-Hub-Cache auf Google Drive
    # für Tokenizer oder zusätzliche kleine Hub-Dateien verwenden.
    eval_env = TEST_ENV.copy()
    eval_env["HF_HOME"] = "/content/robobarista_clean_eval_hf_cache"
    print("Starte offizielle Policy-Evaluation:", task, "|", n_episodes, "Episode(n)", flush=True)
    exit_code = -1
    try:
        with log_path.open("w", encoding="utf-8") as output:
            proc = subprocess.Popen(
                cmd, env=eval_env, cwd="/content/lerobot",
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, bufsize=1,
            )
            for line in proc.stdout:
                print(line, end="")
                output.write(line)
                output.flush()
            exit_code = proc.wait()
    finally:
        if USE_GOOGLE_DRIVE:
            dest = RESULTS / name
            try:
                shutil.copytree(result_dir, dest, dirs_exist_ok=True)
                print("Ergebnisse/Log im Google Drive:", dest)
            except Exception as exc:
                print("⚠️ Drive-Sicherung fehlgeschlagen:", exc)
    if exit_code != 0:
        raise RuntimeError(
            f"Evaluation fehlgeschlagen, Code {exit_code}. Log: {log_path}"
        )
    print("✅ Evaluation erfolgreich beendet. Ergebnisse:", result_dir)
    return result_dir

last_result = evaluate("CloseFridge", n_episodes=1)


## 8. Ergebnisvideo ansehen

Das Video zeigt, wie die **tatsächlich trainierte Policy** den Roboter bewegt – kein abgespieltes RoboCasa-Demonstrationsvideo. Die Aufgaben-Erfolgsquote steht im Log/den Evaluierungsergebnissen. Wenn kein MP4 vorliegt, ist das noch kein Beweis, dass keine Simulation lief.

**Colab-Kompatibilität:** Die Anzeige nutzt ein HTML5-Video statt `IPython.display.Video`, um den `NoneType`-Fehler auf Python 3.13 zu vermeiden. Die vorhandenen Rollouts müssen dafür nicht wiederholt werden.


In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import base64

# Vermeidet den NoneType-Fehler in IPython.display.Video unter Colab/Python 3.13.
# Es wird die echte MP4 der KI-Evaluation angezeigt, keine Zufallsaktionen.
videos = sorted(Path(last_result).rglob("*.mp4"))
if videos:
    video_path = videos[0]
    print("Zeige Video:", video_path)
    print(f"Dateigröße: {video_path.stat().st_size / 1024**2:.1f} MB")
    video_b64 = base64.b64encode(video_path.read_bytes()).decode("ascii")
    display(HTML(f"""
    <video controls playsinline width="720" style="max-width:100%">
      <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
      Dein Browser unterstützt die Videoanzeige nicht.
    </video>
    """))
else:
    print("Kein MP4 im Ergebnisverzeichnis gefunden.")
    print("Dateien:", [str(p.relative_to(last_result)) for p in list(last_result.rglob("*"))[:30]])


## 8b. **Kurztest mit 100 Schritten: Geschwindigkeit + Kamera-Positionssensitivität**

**Diesen Test vor weiteren 900-Schritte-Rollouts ausführen.** Er nutzt dieselbe echte vortrainierte SmolVLA-Policy und dieselbe RoboCasa-Aufgabe, jedoch **maximal 100 statt 900 Simulationsschritte**. Keine manuell programmierten Bewegungen und keine zufälligen Aktionen.

Wir messen getrennt **(a)** `env.step` inklusive Physik und Kamera-Rendering, **(b)** `policy.select_action` inklusive neuer SmolVLA-Berechnungen und **(c)** übrige Laufzeit. Zurzeit liegt die gemessene Gesamtgeschwindigkeit nur bei rund 2 statt 20 Schritten pro Sekunde. Die Auswertung ist zunächst ein Profil, kein Geschwindigkeits-Fix.

**Positionsfrage:** Wir speichern die drei **echten Kameraeingaben** bei Schritt 0, 25, 50, 75 und 100 als Übersicht. Zusätzlich lesen wir aus RoboCasa die **tatsächlichen Türgelenkwinkel** als unabhängige Referenz aus. Schließlich erzeugt die Diagnose einmal zwei Vorhersagen derselben KI mit identischem Zufallsstart: normale Eingangsbilder vs. ein **horizontal um 12,5 % verschobenes Bild der rechten Außenkamera**. Eine Änderung der KI-Aktionen bedeutet *Bildsensitivität*, beweist jedoch **keine** korrekte räumliche Erkennung des Kühlschrankgriffs. Dazu benötigen wir später kontrollierte 3D-Positionsänderungen mehrerer Szenen. Der eigentliche Bewegungsrollout bleibt unverändert; die Bildmanipulation dient nur diesem separaten Vergleich vor dem ersten Schritt.

Es gibt ein Video, eine Kontaktübersicht der drei Kameraansichten, `diagnostics.json` mit Türwinkeln und den Laufzeitanteilen sowie `run.log`. Dateien werden auf Google Drive gesichert, sofern aktiviert. **Nach dem Diagnose-Test**: Video und Kameraübersicht gemeinsam prüfen, **erst dann** weitere Episoden starten.

**Fix vom 20.10.:** Der erste Diagnoseversuch brach vor dem ersten Simulationsschritt ab, weil das Diagnose-Skript direkt in `/content` lag. Python interpretierte `/content/robosuite` dann als gleichnamiges, unvollständiges Namespace-Paket (`robosuite.__file__ == None`). Der normale `lerobot-eval` hatte dieses Problem nicht. Die Skriptdatei liegt jetzt im separaten Ergebnisordner und prüft Paketpfade, bevor das Modell geladen wird.


In [ ]:
# 100-Schritte-Test: unveränderte trainierte Policy, echtes LeRobot-Rollout.
# Komponenten-Timing + drei Kameraansichten + MuJoCo-Türwinkel
# + reproduzierbarer Bildpositions-Reaktionstest.
import os, subprocess, shutil
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image, ImageDraw
from IPython.display import display
import json

assert Path(MODEL_LOCAL_DIR, "model.safetensors").is_file(), "Abschnitt 6 zuerst ausführen."
assert Path(BASE_VLM_LOCAL_DIR, "model.safetensors").is_file(), "Abschnitt 6b zuerst ausführen."

DIAG_ROOT = Path("/content/robobarista_runs") / (
    "CloseFridge_100step_diagnostic_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
)
DIAG_ROOT.mkdir(parents=True, exist_ok=True)
# Absolute Skriptdatei darf NICHT direkt unter /content liegen:
# Python setzt den Skriptordner an sys.path[0]. Der Ordner /content/robosuite
# wäre dort sonst ein Namespacemodul ohne __file__ und verdeckt das installierte
# robosuite-Paket (unsere 0-Schritte-Fehlermeldung).
DIAG_PY = DIAG_ROOT / "robobarista_diag_100.py"
DIAG_PY.write_text("\"\"\"\nRoboBarista Diagnose: offizielle LeRobot-Evaluation, maximal 100 Schritte.\nKeine eigenen Bewegungen: alle Aktionen kommen unverändert aus SmolVLA.\nInstrumentierung: Kameraframes, MuJoCo Türwinkel, Laufzeit und Bildpositions-Probe.\n\"\"\"\nimport sys\nimport os\nimport json\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom PIL import Image\n\nROOT = Path(os.environ[\"ROBOBARISTA_DIAG_DIR\"])\nROOT.mkdir(parents=True, exist_ok=True)\nCAM_DIR = ROOT / \"input_cameras\"\nCAM_DIR.mkdir(parents=True, exist_ok=True)\n\nfrom lerobot.envs.robocasa import RoboCasaEnv\nfrom lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy\nfrom lerobot.utils.constants import ACTION\n\nD = {\n    \"steps\": 0,\n    \"camera_captures\": [],\n    \"env_step_s\": [],\n    \"policy_select_s\": [],\n    \"fresh_action_chunk_steps\": [],\n    \"door_states\": [],\n    \"camera_shift_probe\": None,\n    \"probe_error\": None,\n    \"start_time\": None,\n    \"end_time\": None,\n}\nold_obs = RoboCasaEnv._format_raw_obs\nold_env_step = RoboCasaEnv.step\nold_select = SmolVLAPolicy.select_action\n\ndef door_ground_truth(adapter):\n    try:\n        kitchen = adapter._env.env\n        fridge = getattr(kitchen, \"fxtr\", None)\n        if fridge is None:\n            return {\"available\": False, \"reason\": \"kitchen.fxtr not available\"}\n        joints = getattr(fridge, \"_fridge_door_joint_names\", [])\n        sim = kitchen.sim\n        positions = {}\n        for joint in joints:\n            addr = sim.model.get_joint_qpos_addr(joint)\n            if isinstance(addr, tuple):\n                addr = addr[0]\n            positions[joint] = float(sim.data.qpos[addr])\n        closed = bool(fridge.is_closed(env=kitchen))\n        return {\"available\": True, \"joint_radians\": positions,\n                \"is_closed\": closed}\n    except Exception as ex:\n        return {\"available\": False, \"reason\": repr(ex)}\n\ndef obs_with_snapshots(self, raw):\n    obs = old_obs(self, raw)\n    step = D[\"steps\"]\n    if step in (0, 25, 50, 75, 100):\n        saved = {}\n        for cam, img in obs.get(\"pixels\", {}).items():\n            fn = CAM_DIR / (f\"step_{step:03d}_{cam}.png\")\n            Image.fromarray(np.asarray(img, dtype=np.uint8)).save(fn)\n            saved[cam] = str(fn)\n        if saved:\n            D[\"camera_captures\"].append({\"step\": step, \"files\": saved})\n    return obs\n\ndef timed_step(self, action):\n    t0 = time.perf_counter()\n    result = old_env_step(self, action)\n    elapsed = time.perf_counter() - t0\n    D[\"env_step_s\"].append(elapsed)\n    D[\"steps\"] += 1\n    obs, reward, term, trunc, info = result\n    D[\"door_states\"].append({\n        \"step\": D[\"steps\"],\n        \"door\": door_ground_truth(self) if not bool(term) else {\"available\": False, \"reason\": \"episode finished / env auto-reset\"},\n        \"reward\": float(reward),\n        \"success\": bool(info.get(\"is_success\", False)),\n    })\n    D[\"end_time\"] = time.perf_counter()\n    return result\n\ndef compare_camera_shift(policy, batch):\n    \"\"\"Reproduzierbarer einmaliger Kontrolltest: nur Kamerabild 3 verschieben.\n    Kein Beweis für Türerkennung; nur Reaktion der Aktionsvorhersage auf Bildgeometrie.\n    \"\"\"\n    img_keys = [\n        k for k, v in batch.items()\n        if k.startswith(\"observation.images.\") and torch.is_tensor(v)\n    ]\n    preferred = \"observation.images.camera3\"\n    if preferred in img_keys:\n        cam = preferred\n    elif img_keys:\n        cam = img_keys[-1]\n    else:\n        return {\"available\": False, \"reason\": \"No image tensors reached policy\"}\n    shifted_batch = dict(batch)\n    x = batch[cam]\n    shift_px = max(1, int(x.shape[-1] * 0.125))\n    # Künstliche Bildverschiebung, nicht eine echte 3D-Positionsänderung.\n    shifted_batch[cam] = torch.roll(x, shifts=shift_px, dims=-1)\n    devices = [torch.cuda.current_device()] if torch.cuda.is_available() else []\n    try:\n        with torch.inference_mode():\n            with torch.random.fork_rng(devices=devices):\n                torch.manual_seed(73914)\n                a0 = policy.predict_action_chunk(dict(batch)).detach().float().cpu()\n            policy.reset()\n            with torch.random.fork_rng(devices=devices):\n                torch.manual_seed(73914)\n                a1 = policy.predict_action_chunk(dict(shifted_batch)).detach().float().cpu()\n            policy.reset()\n        diff = a1 - a0\n        return {\n            \"available\": True, \"camera\": cam, \"horizontal_shift_px\": shift_px,\n            \"relative_mean_absolute_change\": float(\n                diff.abs().mean() / (a0.abs().mean() + 1e-8)\n            ),\n            \"rms_action_difference\": float(torch.sqrt((diff ** 2).mean())),\n            \"max_action_difference\": float(diff.abs().max()),\n            \"action_shape\": list(a0.shape),\n            \"note\": \"Künstliche Bildpositionsänderung, keine Objektlokalisierungs-Messung\",\n        }\n    finally:\n        # Das eigentliche 100-Schritte-Rollout beginnt mit leerem Aktions-Cache.\n        policy.reset()\n\ndef timed_select(self, batch, *args, **kwargs):\n    if D[\"start_time\"] is None:\n        D[\"start_time\"] = time.perf_counter()\n    if D[\"camera_shift_probe\"] is None and D[\"probe_error\"] is None:\n        try:\n            tprobe = time.perf_counter()\n            D[\"camera_shift_probe\"] = compare_camera_shift(self, batch)\n            D[\"camera_shift_probe\"][\"elapsed_s\"] = round(time.perf_counter() - tprobe, 3)\n            print(\"\\n[BILDTEST] Kameraeinfluss:\", D[\"camera_shift_probe\"], flush=True)\n        except Exception as exc:\n            D[\"probe_error\"] = repr(exc)\n            self.reset()\n            print(\"\\n[BILDTEST] Nicht auswertbar:\", repr(exc), flush=True)\n    if len(self._queues[ACTION]) == 0:\n        D[\"fresh_action_chunk_steps\"].append(D[\"steps\"])\n    if torch.cuda.is_available():\n        torch.cuda.synchronize()\n    t0 = time.perf_counter()\n    result = old_select(self, batch, *args, **kwargs)\n    if torch.cuda.is_available():\n        torch.cuda.synchronize()\n    D[\"policy_select_s\"].append(time.perf_counter() - t0)\n    return result\n\nRoboCasaEnv._format_raw_obs = obs_with_snapshots\nRoboCasaEnv.step = timed_step\nSmolVLAPolicy.select_action = timed_select\n\nargs = [\n    \"--policy.path=\" + os.environ[\"ROBOBARISTA_POLICY_DIR\"],\n    \"--policy.vlm_model_name=\" + os.environ[\"ROBOBARISTA_VLM_DIR\"],\n    \"--env.type=robocasa\",\n    \"--env.task=CloseFridge\",\n    \"--env.episode_length=100\",\n    \"--eval.batch_size=1\",\n    \"--eval.n_episodes=1\",\n    \"--eval.use_async_envs=false\",\n    \"--policy.device=cuda\",\n    \"--seed=1000\",\n    \"--output_dir=\" + str(ROOT / \"eval_output\"),\n    '--rename_map={\"observation.images.robot0_agentview_left\":\"observation.images.camera1\",\"observation.images.robot0_eye_in_hand\":\"observation.images.camera2\",\"observation.images.robot0_agentview_right\":\"observation.images.camera3\"}',\n]\nsys.argv = [\"lerobot-eval\", *args]\n\ntry:\n    from lerobot.scripts.lerobot_eval import main\n    main()\nfinally:\n    wall = (\n        D[\"end_time\"] - D[\"start_time\"]\n        if D[\"end_time\"] is not None and D[\"start_time\"] is not None\n        else None\n    )\n    step_s = sum(D[\"env_step_s\"])\n    policy_s = sum(D[\"policy_select_s\"])\n    total_step = D[\"steps\"]\n    D[\"timing_summary\"] = {\n        \"steps\": total_step,\n        \"measured_rollout_seconds\": wall,\n        \"env_step_total_seconds\": step_s,\n        \"env_step_avg_ms\": 1000 * step_s / max(1, total_step),\n        \"policy_select_total_seconds\": policy_s,\n        \"policy_select_avg_ms\": 1000 * policy_s / max(1, len(D[\"policy_select_s\"])),\n        \"policy_replanned_at_steps\": D[\"fresh_action_chunk_steps\"],\n        \"other_seconds_including_preprocessing_and_recording\":\n            max(0, wall - step_s - policy_s) if wall is not None else None,\n    }\n    outfile = ROOT / \"diagnostics.json\"\n    outfile.write_text(json.dumps(D, indent=2, ensure_ascii=False), encoding=\"utf-8\")\n    print(\"\\n\" + \"=\" * 60)\n    print(\"RoboBarista 100-Schritte-Diagnose:\", outfile)\n    print(json.dumps(D[\"timing_summary\"], indent=2, ensure_ascii=False))\n    print(\"Kamera-Schnappschüsse:\", len(D[\"camera_captures\"]))\n    print(\"Door ground truth (letzte Messung):\", D[\"door_states\"][-1] if D[\"door_states\"] else None)\n    print(\"Kameraeinfluss-Test:\", D[\"camera_shift_probe\"] or D[\"probe_error\"])\n    print(\"=\" * 60, flush=True)\n", encoding="utf-8")

env = TEST_ENV.copy()
env.update({
    "ROBOBARISTA_DIAG_DIR": str(DIAG_ROOT),
    "ROBOBARISTA_POLICY_DIR": str(MODEL_LOCAL_DIR),
    "ROBOBARISTA_VLM_DIR": str(BASE_VLM_LOCAL_DIR),
    "HF_HOME": "/content/robobarista_diagnostic_clean_hf_cache",
})
# Fehlgeschlagene Paketauflösung erkennen, BEVOR die großen KI-Gewichte geladen werden.
package_check = (
    "import robosuite, robocasa, pathlib; "
    "assert robosuite.__file__, 'robosuite namespace-shadow detected'; "
    "assert robocasa.__file__, 'robocasa namespace-shadow detected'; "
    "assert pathlib.Path(robosuite.__file__).is_file(); "
    "print('Paketpfade geprüft:', robosuite.__file__, robocasa.__file__)"
)
subprocess.run(
    [PY, "-c", package_check], env=env,
    cwd="/content/lerobot", check=True,
)
log = DIAG_ROOT / "run.log"
print("Diagnose startet. Maximal 100 Schritte.", flush=True)
exit_code = -1
try:
    with log.open("w", encoding="utf-8") as f:
        proc = subprocess.Popen(
            [PY, "-u", str(DIAG_PY)],
            cwd="/content/lerobot",
            env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
            f.write(line)
            f.flush()
        exit_code = proc.wait()
finally:
    if USE_GOOGLE_DRIVE:
        target = RESULTS / DIAG_ROOT.name
        try:
            shutil.copytree(DIAG_ROOT, target, dirs_exist_ok=True)
            print("Gesichert in Google Drive:", target)
        except Exception as error:
            print("Drive-Sicherung fehlgeschlagen:", repr(error))

report = DIAG_ROOT / "diagnostics.json"
if report.is_file():
    d = json.loads(report.read_text())
    print("\n=== Geschwindigkeitsmessung ===")
    for k, v in d["timing_summary"].items():
        print(f"{k}: {v}")
    print("\n=== Einfluss künstlicher Bildverschiebung auf KI-Aktionen ===")
    print(json.dumps(d.get("camera_shift_probe") or {"error": d.get("probe_error")}, indent=2, ensure_ascii=False))
    angles = [(e["step"], e["door"]) for e in d["door_states"] if e["door"].get("available")]
    print("\nTürwinkel (Anfang/Ende im 100-Schritte-Test):")
    print(angles[:1], "→", angles[-1:] if angles else [])
    # Zeige keine leere Bildmatrix, wenn vor dem ersten Schritt etwas fehlschlug.
    if not d.get("camera_captures"):
        print("Noch keine Kamerabilder: 0 Simulationsschritte, bitte Laufzeitfehler prüfen.")
    else:
        print("Gespeicherte Kamera-Snapshots:", len(d["camera_captures"]))
        # Übersicht der tatsächlichen Kamera-Eingabebilder für fünf Zeitpunkte
        cams = ["robot0_agentview_left", "robot0_eye_in_hand", "robot0_agentview_right"]
        sizes = (256, 256)
        step_labels = (0, 25, 50, 75, 100)
        panel = Image.new("RGB", (768, len(step_labels) * 285), "white")
        draw = ImageDraw.Draw(panel)
        for row, step in enumerate(step_labels):
            for col, cam in enumerate(cams):
                p = DIAG_ROOT / "input_cameras" / f"step_{step:03d}_{cam}.png"
                if p.is_file():
                    panel.paste(Image.open(p).convert("RGB"), (col*256, row*285+23))
                draw.text((col*256+8,row*285+5), f"Step {step}: {cam.replace('robot0_','')}", fill="black")
        panel_path = DIAG_ROOT / "camera_comparison.jpg"
        panel.save(panel_path, quality=92)
        display(panel)
        print("Kameravergleich gespeichert:", panel_path)
else:
    print("Kein Diagnosebericht; bitte die letzten Logzeilen hier teilen:", log)

if exit_code:
    raise RuntimeError(f"100-Schritte-Test fehlgeschlagen, Exit-Code {exit_code}; Log: {log}")
print("✅ 100-Schritte-Test abgeschlossen:", DIAG_ROOT)


## 9. Nächster Test: Pick-and-Place mit drei unterschiedlichen Episoden

**Erst starten, wenn CloseFridge technisch funktioniert.** Drei Episoden dienen zunächst als Funktionsprobe, nicht als robuste Erfolgsstatistik. Wir prüfen danach im Video, ob die Objektpositionen wirklich variierten. Das **aktive Verschieben eines Objekts während der Bewegung** und die **Messung von Positions-/Orientierungsfehlern** werden als eigene Versuchssteuerung ergänzt, ohne selbst Greifbewegungen zu programmieren.

In [ ]:
pickplace_result = evaluate("PickPlaceCounterToStove", n_episodes=3)


## 10. Was als nächstes kommt

Nachdem das Basissetup funktioniert, erstellen wir ein getrenntes Experiment für **(a)** randomisierte Startpositionen, **(b)** Störungen während eines Greifversuchs und **(c)** messbare Endpositionen/Orientierungen inklusive Erfolgsrate. Danach erproben wir eine zusammenhängende Sequenz mehrerer Objekte. Das Standard-`lerobot-eval` verknüpft verschiedene RoboCasa-Aufgaben **nicht automatisch zu einer einzigen Manipulationsfolge**.

**Fehler?** Bitte nur die Ausgabe der ersten fehlgeschlagenen Zelle schicken. Keine Installationszellen auf Verdacht mehrfach ausführen und insbesondere nicht auf LeRobot 0.3.3 downgraden.

**Sicherheit:** Dieser Workflow ist Simulation-only. Für einen realen Barista-Roboter brauchen wir zusätzlich verifizierte Sicherheitsgrenzen sowie Kontakt-/Kraft- und Temperaturüberwachung.
